# 2 — Covalently modify a protein, watch it relax, and simulate it
1. **Load a protein** with full chemistry (notebook 1 wrote the input).
2. **Build a fragment from scratch** as star-sited SMILES. The `*` marks
   where the bond forms.
3. **Deprotonate the attachment site.** The reactive form of a lysine side
   chain is the neutral amine, not the ammonium ion.
4. **Attach them.** One call places the fragment, removes one hydrogen per
   side, and forms the bond.
5. **Look at the result**, then **relax the fragment as a movie**. The
   protein stays fixed and only the fragment moves.
6. **Export** the modified PDB plus one bond record per new bond.
7. **Parameterize.** Those two outputs are all openff-pablo needs, and from
   there Interchange and OpenMM take over.

Run notebook 1 first, so `1ubq_protonated.pdb` exists.

In [1]:
from mbuild.biopolymers import Protein, prepare_fragment

import demo_utils

protein = Protein("1ubq_protonated.pdb")
print(len(list(protein.residues())), "residues | net formal charge:",
      protein.net_formal_charge)

76 residues | net formal charge: 0


## 1. Prepare the fragment

Write the fragment as SMILES with a `*` at the atom that forms the bond.
`prepare_fragment` turns the star into a hydrogen, keeps the charges written
in the SMILES, gives every atom a PDB-style name, and records the bond site
in `link_atoms`. Here: an octanoyl group, starred at the carbonyl carbon.

Choose the residue code with care. A fragment code must not match a code
that the wwPDB Chemical Component Dictionary already assigns. This demo
checked `OC8` against the RCSB and found it unassigned. A code that
collides, `OCT` for n-octane for example, lets a downloaded CCD template
take the place of the fragment definition, and the modified structure then
fails to read back.

In [2]:
fragment = prepare_fragment("*C(=O)CCCCCCC", "OC8")
print("bond site:", fragment.link_atoms)
print(fragment.n_particles, "atoms |",
      [particle.name for particle in fragment.particles()][:6], "...")

bond site: {'1': 'C1'}
25 atoms | ['H1', 'C1', 'O1', 'C2', 'C3', 'C4'] ...


## 2. Deprotonate the attachment site

At pH 7 a lysine side chain carries a protonated ammonium group. That group
has no lone pair, so it does not attack the acyl carbon and it is not the
reactive form of the side chain. A conjugation reaction is run under
conditions that favour the neutral free amine, and the reaction replaces one
N-H bond of that amine with the new bond.

`Protein.deprotonate` removes one proton from the named atom and re-matches
the residue to the Chemical Component Dictionary variant that describes the
result. The template, the residue formal charge and the per-atom formal
charges then all agree with the structure.

In [3]:
before = protein.net_formal_charge
protein.deprotonate(63, "NZ", chain_id="A")

lysine = protein.get_residue(63, chain_id="A")
nz = protein.get_atom(63, "NZ", chain_id="A")
print("LYS 63 formal charge:", lysine.formal_charge,
      "| bonded to NZ:", sorted(atom.name for atom in nz.direct_bonds()))
print("net formal charge:", before, "->", protein.net_formal_charge)

LYS 63 formal charge: 0 | bonded to NZ: ['CE', 'HZ1', 'HZ2']
net formal charge: 0 -> -1


## 3. Attach it

Name the protein site by residue number and atom name. The fragment already
knows its own bond site from the `*`. One hydrogen leaves each side, ports
along the two removed-hydrogen vectors align the fragment, and the bond
forms.

The net formal charge stays at -1 across the new bond. The bond takes the
place of an N-H bond, so the nitrogen keeps its charge of zero and the
product is a neutral secondary amide. The charge fell from 0 to -1 at the
deprotonation, when the site lost its proton, so each modified lysine lowers
the net charge of the protein by one.

`attach` relaxes the fragment when it lands too close to the protein. Here
`relax=False` turns that off, because the next steps show the relaxation as
a movie. The warning about close atoms is the expected result of that
choice.

In [4]:
record = protein.attach(fragment, resnum=63, atom_name="NZ",
                        chain_id="A", relax=False)
print("new bond:", record.residue1.name, record.atom1_name, "-",
      record.residue2.name, record.atom2_name)
print("leaving hydrogens:", record.leaving1, record.leaving2)
print("LYS 63 formal charge:",
      protein.get_residue(63, chain_id="A").formal_charge,
      "| net formal charge:", protein.net_formal_charge)

2026-09-04 14:38:57,769 - mbuild.biopolymers.protein - WARNING - 1 atoms of the attached fragment sit within 1.0 A of existing atoms (closest: 0.43 A). Relax the structure before simulating (e.g. relax_fragments(), which holds the protein fixed).


new bond: LYS NZ - OC8 C1
leaving hydrogens: ('HZ1',) ('H1',)
LYS 63 formal charge: 0 | net formal charge: -1


## 4. Look at the modification

`Protein.save_pdb` writes the file that every downstream loader reads, so the
view is built from that text. `show_protein` takes the linkage residues from
`bond_records()` and the fragment residues from the HETATM flag, so it needs
no further argument.

The protein is a grey cartoon, the modified LYS 63 is cyan sticks, the new
OC8 residue is green sticks, and the view centers on the fragment.

In [5]:
demo_utils.show_protein(protein)

NGLWidget(layout=Layout(height='500px', width='700px'))

## 5. Relax the fragment as a movie

`protein.relax_fragments()` is the one-call form: it minimizes every HETATM
residue with mBuild's generic parameters while every other atom carries zero
mass, so the protein coordinates do not change.

`demo_utils.relax_movie` runs that same minimization in short bursts and
saves the coordinates after each burst, so the frames become a movie. Frame
0 is the rigid placement, before any minimization. Most of the motion
happens in the first frames, so the movie shows the fragment swing out of
the clash and settle.

The colors mean the same as in the picture above. Press play in the
widget below.

In [6]:
import numpy as np

movie = demo_utils.relax_movie(protein, "octanoyl_relax.pdb",
                               n_frames=40, steps_per_frame=5)
path, frames, energies = movie
step = np.linalg.norm(frames[1:] - frames[:-1], axis=2).max(axis=1)
print(path, "|", len(frames), "frames |", frames.shape[1], "atoms")
print("largest atom step per frame (A), frames 1-3:", np.round(step[:3], 2))
print("potential energy (kJ/mol): first", round(float(energies[0])),
      "-> last", round(float(energies[-1])))

view = demo_utils.show_movie(movie, protein=protein)
view

octanoyl_relax.pdb | 40 frames | 1253 atoms
largest atom step per frame (A), frames 1-3: [9.99 5.14 1.25]
potential energy (kJ/mol): first 285350 -> last 160614


NGLWidget(layout=Layout(height='500px', width='700px'), max_frame=39)

A GIF of the movie needs a live browser front end, because NGLView asks
the browser for each picture. `save_gif` writes a file only when the
environment variable `DEMO_RENDER` is set, so this cell does nothing in a
headless run.

In [7]:
demo_utils.save_gif(view, frames, "octanoyl_relax.gif")

## 6. Write the modified PDB and the bond records

`save_pdb` writes a standards-conformant file: residue names, real PDB
residue numbers, chain identifiers, HETATM for the fragment, TER after each
chain, and CONECT records only for the bonds that residue adjacency cannot
imply.

`bond_records()` returns one plain dict per new bond. It names the two
residues, the two bonded atoms, the hydrogens that left each side, and the
bond order. That is everything a downstream loader needs to know about the
modification.

In [8]:
protein.save_pdb("1ubq_octanoyl.pdb", overwrite=True)
records = protein.bond_records()
records

[{'residue_names': ('LYS', 'OC8'),
  'residue_numbers': (63, 77),
  'chain_ids': ('A', 'A'),
  'icodes': ('', ''),
  'atom_names': ('NZ', 'C1'),
  'leaving_atoms': (['HZ1'], ['H1']),
  'bond_order': 1}]

## 7. Ingest with OpenFF Pablo

Pablo reads the PDB file against residue templates. It knows the CCD
residues, so it needs two things from us: a definition for OC8, and a
declaration of the crosslink.

`demo_utils.pablo_crosslink_kwargs` renames the keys of one bond record into
Pablo's `with_crosslink` vocabulary. No chemistry is added: the record
already holds it.

In [9]:
from openff.pablo import STD_CCD_CACHE, ResidueDefinition, topology_from_pdb
from openff.toolkit import Molecule
from rdkit import Chem

# Build the OC8 definition from the SAME starred SMILES, with the star
# replaced by hydrogen, exactly as prepare_fragment did. The atom order
# then matches, so the names transfer by position.
star = Chem.RWMol(Chem.MolFromSmiles("*C(=O)CCCCCCC"))
for atom in star.GetAtoms():
    if atom.GetAtomicNum() == 0:
        atom.SetAtomicNum(1)
mol = star.GetMol()
Chem.SanitizeMol(mol)
offmol = Molecule.from_rdkit(Chem.AddHs(mol), allow_undefined_stereo=True)

# Zip against the pristine `fragment`, not against the OC8 residue inside
# the protein: attach() cloned the fragment and removed H1 from the clone,
# so the residue holds one atom less and every name after H1 would shift.
for atom, particle in zip(offmol.atoms, fragment.particles()):
    atom.name = particle.name

library = STD_CCD_CACHE.with_(
    {"OC8": [ResidueDefinition.from_molecule(offmol, residue_name="OC8")]}
).with_crosslink(**demo_utils.pablo_crosslink_kwargs(records[0]))

topology = topology_from_pdb("1ubq_octanoyl.pdb", residue_library=library)
molecule = topology.molecule(0)
print(molecule.n_atoms, "atoms | net charge:", molecule.total_charge)

1253 atoms | net charge: -1.0 elementary_charge


## 8. Assign force-field parameters

We use the OpenFF ecosystem to assign parameters.

`demo_charges.assign_split_charges` gives every atom of the conjugate one
partial charge, from one of two models. Every atom of a standard residue
keeps its Amber **ff14SB** library charge, read from the unmodified protein.
The fragment and the modified lysine take **NAGL am1bcc graph charges**
(`openff-gnn-am1bcc-0.1.0-rc.3`), computed on a capped local model of the
modification site. The seam between the two models therefore lies on the
peptide bonds of the modified residue. The two sets do not sum to the formal
charge exactly, so the function spreads the small residual over the atoms of
that scope and prints it.

The table reports, per atom of the site, the ff14SB charge, the graph charge,
the final charge, and the difference between the first two. A dash marks an
atom that ff14SB does not describe.

In [10]:
import demo_charges

# The protein before the modification supplies the ff14SB library
# charges. It is the file that notebook 1 wrote.
unmodified_topology = topology_from_pdb("1ubq_protonated.pdb")

conjugate = demo_charges.assign_split_charges(
    molecule, unmodified_topology, fragment_resname="OC8",
    resnum=63, chain_id="A")

charges of LYS63 and OC8
 residue   atom    ff14SB      NAGL     final     delta
   LYS63      N   -0.3479   -0.5668   -0.5658   -0.2189
   LYS63      H   +0.2747   +0.3191   +0.3201   +0.0444
   LYS63     CA   -0.2400   +0.0441   +0.0452   +0.2841
   LYS63     HA   +0.1426   +0.0847   +0.0857   -0.0579
   LYS63      C   +0.7341   +0.6360   +0.6370   -0.0981
   LYS63      O   -0.5894   -0.6021   -0.6011   -0.0127
   LYS63     CB   -0.0094   -0.0988   -0.0978   -0.0894
   LYS63    HB2   +0.0362   +0.0603   +0.0614   +0.0241
   LYS63    HB3   +0.0362   +0.0603   +0.0614   +0.0241
   LYS63     CG   +0.0187   -0.0784   -0.0774   -0.0971
   LYS63    HG2   +0.0103   +0.0472   +0.0482   +0.0369
   LYS63    HG3   +0.0103   +0.0472   +0.0482   +0.0369
   LYS63     CD   -0.0479   -0.1061   -0.1051   -0.0582
   LYS63    HD2   +0.0621   +0.0490   +0.0500   -0.0131
   LYS63    HD3   +0.0621   +0.0490   +0.0500   -0.0131
   LYS63     CE   -0.0143   +0.0984   +0.0994   +0.1127
   LYS63    HE2   +0.11

## 9. Solvate and run MD with OpenMM

`demo_charges.parameterize_with_preset_charges` builds the Interchange with
the split charges passed in as preset charges. It then reads the charges back
out of the Interchange and compares them to the array it passed in. That
check is needed because Sage 2.3.0 carries its own NAGLCharges handler, which
would otherwise compute new charges for the whole conjugate.

In [11]:
from openff.interchange.components._packmol import UNIT_CUBE, pack_box
from openff.toolkit import Topology
from openff.units import unit as off_unit

water = Molecule.from_smiles("O")
water.generate_conformers(n_conformers=1)
for atom in water.atoms:
    atom.metadata["residue_name"] = "HOH"

solvated = pack_box([water], [1500],
                    solute=Topology.from_molecules([conjugate]),
                    target_density=0.95 * off_unit.gram / off_unit.milliliter,
                    box_shape=UNIT_CUBE,
                    tolerance=2.0 * off_unit.angstrom)
print("solvated:", solvated.n_atoms, "atoms")

interchange = demo_charges.parameterize_with_preset_charges(
    solvated, conjugate)
print("parameterized:", interchange.topology.n_atoms, "atoms")

solvated: 5753 atoms


parameterized: 5753 atoms


In [12]:
import openmm
from openmm import unit

simulation = interchange.to_openmm_simulation(
    integrator=openmm.LangevinMiddleIntegrator(
        300 * unit.kelvin, 1.0 / unit.picosecond, 2.0 * unit.femtosecond),
)
simulation.minimizeEnergy(maxIterations=200)
simulation.context.setVelocitiesToTemperature(300 * unit.kelvin)
for block in range(2):
    simulation.step(250)
    state = simulation.context.getState(getEnergy=True)
    print(f"step {(block + 1) * 250}: PE =", state.getPotentialEnergy())
print("MD ran: a covalently modified protein, built in mBuild, in OpenMM.")

step 250: PE = -73227.7168475112 kJ/mol


step 500: PE = -73400.15533647139 kJ/mol
MD ran: a covalently modified protein, built in mBuild, in OpenMM.


## Recap

- **mBuild** is responsible for the coordinates and topology: strict protein loading, fragment
  definition from SMILES, deprotonation of the attachment site, covalent
  attachment, fragment relaxation with the protein fixed, and
  chemistry-complete export.
- **OpenFF** is responsible for the parameters: Pablo ingestion from the PDB file plus
  one bond record, ff14SB library charges on the standard residues with NAGL
  am1bcc graph charges on the modified site, ff14SB and Sage 2.3.0 through
  Interchange, then OpenMM.


---

## Appendix: a larger fragment

Nothing above is specific to a small fragment. This appendix repeats the
mBuild half with a sulfonated cyanine FRET dye, 106 atoms carrying one
positive and one negative formal charge, on a fresh copy of the protein.
The dye lands with no clash, so it needs no relaxation and no movie.

The dye bonds to the same lysine nitrogen as the octanoyl group, so the site
needs the same deprotonation. The neutral amine is the reactive form here
too, and the nitrogen carries no charge in the product.

The dye code `FCY` is unassigned in the Chemical Component Dictionary
too, so it carries the same guarantee as `OC8`.

It stops at the charges. The parameterization from there is the same as
above.

In [13]:
DYE_SMILES = (
    "CC1(C2=C(C=CC(=C2)S(=O)(=O)O)[N+](=C1C=CC=CC=C3C(C4=C(N3CCCS(=O)(=O)[O-])"
    "C=CC(=C4)S(=O)(=O)O)(C)CCCCC(=O)NCC*)CCCS(=O)(=O)O)C"
)

dye_protein = Protein("1ubq_protonated.pdb")
dye_protein.deprotonate(63, "NZ", chain_id="A")
dye = prepare_fragment(DYE_SMILES, "FCY")
print("dye:", dye.n_particles, "atoms | formal charge:", dye.formal_charge,
      "| bond site:", dye.link_atoms)

dye_record = dye_protein.attach(dye, resnum=63, atom_name="NZ", chain_id="A")
dye_protein.save_pdb("1ubq_FRET.pdb", overwrite=True)
dye_records = dye_protein.bond_records()
print("new bond:", dye_record.atom1_name, "-", dye_record.atom2_name,
      "| leaving:", dye_record.leaving1, dye_record.leaving2)
print("net formal charge:", dye_protein.net_formal_charge)

INFO:mbuild.biopolymers.fragments:Fragment 'Compound' wrapped into residue 'FCY'.


INFO:mbuild.biopolymers.fragments:Renamed atoms of residue FCY to element+index names so they are unique within the residue.


INFO:mbuild.biopolymers.protein:Wrote 1 bond records to 1ubq_FRET.bondrecords.json. Load the modified protein again with Protein("1ubq_FRET.pdb", bond_records="1ubq_FRET.bondrecords.json").


dye: 106 atoms | formal charge: 0 | bond site: {'1': 'C33'}
new bond: NZ - C33 | leaving: ('HZ1',) ('H1',)
net formal charge: -1


`demo_utils.pablo_residue_library` performs the whole of step 7 in one
call: it builds the residue definition from the starred SMILES and adds one
crosslink declaration per bond record. It needs the pristine `dye`, for the
same reason the zip in step 7 did.

The charge split of section 8 then runs unchanged, with `FCY` as the fragment
residue name.

In [14]:
dye_library = demo_utils.pablo_residue_library(
    DYE_SMILES, dye, "FCY", dye_records)
dye_topology = topology_from_pdb("1ubq_FRET.pdb",
                                 residue_library=dye_library)
dye_conjugate = dye_topology.molecule(0)
print(dye_conjugate.n_atoms, "atoms | net charge:",
      dye_conjugate.total_charge)

dye_conjugate = demo_charges.assign_split_charges(
    dye_conjugate, unmodified_topology, fragment_resname="FCY",
    resnum=63, chain_id="A")

1334 atoms | net charge: -1.0 elementary_charge


charges of LYS63 and FCY
 residue   atom    ff14SB      NAGL     final     delta
   LYS63      N   -0.3479   -0.5663   -0.5659   -0.2184
   LYS63      H   +0.2747   +0.3196   +0.3200   +0.0449
   LYS63     CA   -0.2400   +0.0441   +0.0445   +0.2841
   LYS63     HA   +0.1426   +0.0852   +0.0856   -0.0574
   LYS63      C   +0.7341   +0.6365   +0.6369   -0.0976
   LYS63      O   -0.5894   -0.6016   -0.6012   -0.0122
   LYS63     CB   -0.0094   -0.0970   -0.0966   -0.0876
   LYS63    HB2   +0.0362   +0.0601   +0.0605   +0.0239
   LYS63    HB3   +0.0362   +0.0601   +0.0605   +0.0239
   LYS63     CG   +0.0187   -0.0732   -0.0728   -0.0919
   LYS63    HG2   +0.0103   +0.0451   +0.0455   +0.0348
   LYS63    HG3   +0.0103   +0.0451   +0.0455   +0.0348
   LYS63     CD   -0.0479   -0.0810   -0.0805   -0.0331
   LYS63    HD2   +0.0621   +0.0471   +0.0475   -0.0150
   LYS63    HD3   +0.0621   +0.0471   +0.0475   -0.0150
   LYS63     CE   -0.0143   +0.1734   +0.1738   +0.1877
   LYS63    HE2   +0.11

`show_protein` reads a PDB file as well as a `Protein`. A file holds no bond
record and no HETATM flag on a residue object, so the derivation cannot run:
name the linkage residue and the fragment with NGL selections instead. The
colors mean the same as above.

In [15]:
demo_utils.show_protein("1ubq_FRET.pdb", link_selection="63:A",
                        fragment_selection="[FCY]")

NGLWidget(layout=Layout(height='500px', width='700px'))